# Model

In [1]:
import pandas as pd
import numpy as np
import os
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader as DL

import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

from src import model

In [2]:
dir = os.getenv('DIR')
os.getcwd()
os.chdir(dir)
path = os.getcwd()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Teams Data
df = pd.read_csv(os.path.join(path,'data','final','Dataset.csv'))
df['Key'] = df['Key'].astype('str')
df.set_index('Key',inplace=True)

In [22]:
df

,Score,FGM,FGA,FGM3,FGA3,FTM,FTA,OR,DR,Ast,...,Blk,PF,FG%,3P%,FT%,Possessions,NumOT,GamesPlayed,WinRate,RollingScore
Key,,,,,,,,,,,,,,,,,,,,,
1101,0.924,0.701,0.878,0.436,0.648,0.596,0.664,0.491,0.673,0.580,...,0.282,0.672,0.079,0.065,0.117,0.924,0.016,1.274,0.465,0.926
1102,0.905,0.682,0.852,0.466,0.675,0.555,0.631,0.444,0.672,0.583,...,0.275,0.633,0.081,0.066,0.114,0.901,0.009,1.440,0.401,0.915
1103,0.935,0.711,0.881,0.483,0.692,0.589,0.662,0.530,0.696,0.587,...,0.324,0.645,0.081,0.067,0.115,0.918,0.016,1.437,0.660,0.965
1104,0.939,0.715,0.885,0.463,0.678,0.606,0.678,0.549,0.710,0.574,...,0.374,0.640,0.081,0.064,0.116,0.923,0.015,1.455,0.614,0.985
1105,0.910,0.686,0.880,0.408,0.642,0.594,0.678,0.541,0.690,0.548,...,0.330,0.655,0.074,0.058,0.111,0.925,0.013,1.409,0.324,0.935
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3477,0.907,0.691,0.894,0.419,0.653,0.557,0.627,0.491,0.699,0.564,...,0.351,0.616,0.071,0.058,0.118,0.937,0.008,1.031,0.369,0.896
3478,0.877,0.655,0.868,0.414,0.650,0.540,0.597,0.490,0.676,0.537,...,0.322,0.591,0.068,0.058,0.122,0.913,0.012,0.985,0.381,0.883
3479,0.906,0.678,0.874,0.481,0.707,0.560,0.622,0.430,0.653,0.552,...,0.259,0.643,0.073,0.062,0.121,0.931,0.017,0.863,0.366,0.933


In [23]:
teams = model.TeamDataset(df)
teamsdataloader = DL(teams, batch_size=64, shuffle=False)

encoder = model.TeamEncoder(input_dim=df.shape[1])
model.train_encoder(encoder, teamsdataloader)

Epoch 1, Loss: 1.2190
Epoch 2, Loss: 0.0843
Epoch 3, Loss: 0.0587
Epoch 4, Loss: 0.0476
Epoch 5, Loss: 0.0431
Epoch 6, Loss: 0.0455
Epoch 7, Loss: 0.0443
Epoch 8, Loss: 0.0441
Epoch 9, Loss: 0.0423
Epoch 10, Loss: 0.0382
Epoch 11, Loss: 0.0371
Epoch 12, Loss: 0.0354
Epoch 13, Loss: 0.0355
Epoch 14, Loss: 0.0354
Epoch 15, Loss: 0.0343
Epoch 16, Loss: 0.0331
Epoch 17, Loss: 0.0326
Epoch 18, Loss: 0.0323
Epoch 19, Loss: 0.0315
Epoch 20, Loss: 0.0313


In [41]:
# Matchups Data
matchups = pd.read_csv(os.path.join(path,'data','final','Matchups.csv'))

target = matchups[['Key','Win']]
target['TeamAID'] = target['Key'].apply(lambda x: x.split('-')[0])
target['TeamBID'] = target['Key'].apply(lambda x: x.split('-')[1])
target = target[['TeamAID','TeamBID','Win']]

In [42]:
# Storing the Encoder values after training
encoder.eval()

team_embeddings = {}

with torch.no_grad():
    for team_id, stats in df.iterrows():
        stats = torch.tensor(stats.values, dtype=torch.float32).unsqueeze(0).to(device)

        _, emb = encoder(stats)   # (1, d)

        team_embeddings[team_id] = emb.squeeze(0).cpu()


team_ids = sorted(team_embeddings.keys())
team_embeddings = torch.stack([
    team_embeddings[tid] for tid in team_ids
])

In [43]:
df.index.values

array(['1101', '1102', '1103', '1104', '1105', '1106', '1107', '1108',
       '1110', '1111', '1112', '1113', '1114', '1115', '1116', '1117',
       '1119', '1120', '1122', '1123', '1124', '1125', '1126', '1127',
       '1128', '1129', '1130', '1131', '1132', '1133', '1135', '1136',
       '1137', '1138', '1139', '1140', '1141', '1142', '1143', '1144',
       '1145', '1146', '1147', '1148', '1149', '1150', '1151', '1152',
       '1153', '1154', '1155', '1156', '1157', '1158', '1159', '1160',
       '1161', '1162', '1163', '1164', '1165', '1166', '1167', '1168',
       '1169', '1170', '1171', '1172', '1173', '1174', '1175', '1176',
       '1177', '1178', '1179', '1180', '1181', '1182', '1183', '1184',
       '1185', '1186', '1187', '1188', '1189', '1190', '1191', '1192',
       '1193', '1194', '1195', '1196', '1197', '1198', '1199', '1200',
       '1201', '1202', '1203', '1204', '1205', '1206', '1207', '1208',
       '1209', '1210', '1211', '1212', '1213', '1214', '1216', '1217',
      

In [44]:
team_ids.index('1272')

166

In [45]:
# matchups

matchups = model.MatchDatasetEmb(target, team_embeddings)

matchupsdataloader = DL(
    matchups,
    batch_size=64, 
    shuffle=True
)


d_model = team_embeddings.shape[1]

classifier = model.MatchPredictorEmb(d_model)

model.train_classifier(classifier, matchupsdataloader, epochs=10, lr=1e-3)

ValueError: '3379' is not in list

# Prediction

In [ ]:

import sys
import os
import pandas as pd
import numpy as np
from src import model

from dotenv import load_dotenv
load_dotenv()

dir = os.getenv('DIR')
os.getcwd()
os.chdir(dir)
path = os.getcwd()

filename = sys.argv[1]

# Open and read the file
data = pd.read_csv(os.path.join(path,'data','raw',filename))

predictor = classifier

ids = data['ID'].apply(lambda x: x.split('_')).to_list()

preds = []
for i in range(len(data)):
    teamA_id = ids[i][1]
    teamB_id = ids[i][2]
    key = ids[i][0] + '_' + ids[i][1] + '_' + ids[i][2]

    predictions = model.predict(predictor,teamA_id,teamB_id,embedding_matrix=train.team_embeddings)

    preds.append([key,predictions])

preds = pd.DataFrame(preds,columns=['ID','Pred'])

preds.to_csv(os.path.join(path,'data','final',filename))